In [ ]:
import pandas as pd
from monte import train_with_cv, fine_tune_with_cv
from scipy.stats import pearsonr

for LUAD and LUSC, we exclude both cancer types while prior monte training

In [ ]:
# cancer_type = "BRCA"
# exclude_cancer = ["BRCA"]
cancer_type = "LUSC"
exclude_cancer = ["LUAD", "LUSC"]
metric = "CPE"

Load training data for MONTE prior model

In [ ]:
# the parquet file and metadata csv file could be downloaded from Zenodo
df_beta_train = pd.read_parquet("../../data/methylation/train_pan-cancer_beta.parquet")
df_meta_train = pd.read_csv("../../data/methylation/train_pan-cancer_meta.csv")
df_meta_train = df_meta_train.set_index("Barcode", drop=False)
df_meta_train = df_meta_train.loc[df_beta_train.index]

In [ ]:
df_beta_exclude_cancer_train = df_beta_train[df_meta_train["Cancer.type"].isin(exclude_cancer) == False]
df_meta_exclude_cancer_train = df_meta_train[df_meta_train["Cancer.type"].isin(exclude_cancer) == False]

validation data for fine tune

In [ ]:
# the parquet file and metadata csv file could be downloaded from Zenodo
df_beta_val = pd.read_parquet("../../data/methylation/val_pan-cancer_beta.parquet")
df_meta_val = pd.read_csv("../../data/methylation/val_pan-cancer_meta.csv")
df_meta_val = df_meta_val.set_index("Barcode", drop=False)
df_meta_val = df_meta_val.loc[df_beta_val.index]

In [ ]:
df_beta_cancer_val = df_beta_val[df_meta_val["Cancer.type"] == cancer_type]
df_meta_cancer_val = df_meta_val[df_meta_val["Cancer.type"] == cancer_type]

Load test set

In [ ]:
# the parquet file and metadata csv file could be downloaded from Zenodo
df_beta_test = pd.read_parquet("../../data/methylation/test_pan-cancer_beta.parquet")
df_meta_test = pd.read_csv("../../data/methylation/test_pan-cancer_meta.csv")
df_meta_test = df_meta_test.set_index("Barcode", drop=False)
df_meta_test = df_meta_test.loc[df_beta_test.index]

In [ ]:
df_meta_cancer_test = df_meta_test[df_meta_test["Cancer.type"] == cancer_type]
df_beta_cancer_test = df_beta_test.loc[df_meta_cancer_test.index]

### Training MONTE without target cancer type as prior model then fine-tuned with transfer learning

In [ ]:
monte_prior_model = train_with_cv(df_beta_exclude_cancer_train, df_meta_exclude_cancer_train["CPE"])

In [ ]:
monte_fine_tuned = fine_tune_with_cv(monte_prior_model, df_beta_cancer_val, df_meta_cancer_val["CPE"])

In [11]:
df_beta_cancer_corrected = monte_fine_tuned.purify_values(df_beta_cancer_test, alpha=0.05)

Adjusting 54895 probes passing significance threshold of 0.05.


In [12]:
df_beta_cancer_corrected.to_parquet(f"../../data/monte_outputs/probe_correction/{cancer_type}_beta_correction.parquet")

### Correlations between probes and cancer purity

In [13]:
purity_metric = "CPE"
y_metric = df_meta_cancer_test[purity_metric]

In [14]:
probe_ids = df_beta_cancer_test.columns
df_cor = pd.DataFrame(index=probe_ids, columns=["before_correction", "after_correction"])

In [15]:
for probe in probe_ids:
    x_before = df_beta_cancer_test[probe]
    cor_before, _ = pearsonr(x_before, y_metric)
    df_cor.loc[probe, "before_correction"] = cor_before

    x_after = df_beta_cancer_corrected[probe]
    cor_after, _ = pearsonr(x_after, y_metric)
    df_cor.loc[probe, "after_correction"] = cor_after

In [16]:
df_cor.to_csv(f"../../data/monte_outputs/pancancer/monte_{cancer_type}_corrected_probe_correlations.csv")